# Homework 2 - Pattern Recognition and Machine Learning

**Kernel density estimation, bandwidth selection, and density-based classification**

Consider the 1D mixture density

$$f(x) = 0.4\,\mathcal{N}(-1,\,0.5^2) + 0.6\,\mathcal{N}(2,\,1.0^2).$$

KDE estimator:

$$\hat f(x) = \frac{1}{n h}\sum_{i=1}^n K\!\left(\frac{x - x_i}{h}\right),$$

where $K$ is a kernel function and $h$ is the bandwidth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# True mixture parameters
weights = np.array([0.4, 0.6])
mus     = np.array([-1.0, 2.0])
sigmas  = np.array([0.5, 1.0])

def gaussian_pdf_1d(x, mu, sigma):
    return (1.0 / (np.sqrt(2*np.pi) * sigma)) * np.exp(-0.5 * ((x - mu)/sigma)**2)

def true_density(x):
    return (weights[0] * gaussian_pdf_1d(x, mus[0], sigmas[0])
          + weights[1] * gaussian_pdf_1d(x, mus[1], sigmas[1]))

def sample_from_mixture(n):
    z = np.random.choice([0, 1], size=n, p=weights)
    x = np.empty(n)
    x[z == 0] = np.random.normal(mus[0], sigmas[0], size=np.sum(z == 0))
    x[z == 1] = np.random.normal(mus[1], sigmas[1], size=np.sum(z == 1))
    return x

# 1 / Density estimation for a multimodal distribution

## a) Generate random samples for $n = 100, 500, 2000$

In [ ]:
sample_sizes = [100, 500, 2000]
samples_by_n = {n: sample_from_mixture(n) for n in sample_sizes}

for n_cur, s in samples_by_n.items():
    print(f"n = {n_cur:>4}   mean = {s.mean():.3f}   std = {s.std():.3f}")

## b) Gaussian KDE from scratch

$$\hat f(x) = \frac{1}{n h}\sum_{i=1}^n K\!\left(\frac{x - x_i}{h}\right),\qquad
K_G(u) = \frac{1}{\sqrt{2\pi}}\,\exp\!\left(-\tfrac{1}{2}u^2\right).$$

In [ ]:
def gaussian_kernel(u):
    return (1.0 / np.sqrt(2*np.pi)) * np.exp(-0.5 * u**2)

def uniform_kernel(u):
    return 0.5 * (np.abs(u) <= 1)

def epanechnikov_kernel(u):
    vals = 0.75 * (1 - u**2)
    vals[np.abs(u) > 1] = 0.0
    return vals

def triangular_kernel(u):
    return np.maximum(1 - np.abs(u), 0.0)

def kde_estimate(x_eval, samples, h, kernel_fn=gaussian_kernel):
    """General 1D KDE evaluated at x_eval (vectorized)."""
    x_eval = np.asarray(x_eval)
    u = (x_eval[:, None] - samples[None, :]) / h          # (m, n)
    K = kernel_fn(u)                                       # (m, n)
    return K.sum(axis=1) / (len(samples) * h)

## c) Estimate density on a grid for each $n$

We use a Silverman-style bandwidth for the initial visualisation so that the comparison across $n$ depends only on sample size.

In [ ]:
x_grid = np.linspace(-4, 6, 500)
f_true = true_density(x_grid)

def silverman_bandwidth(samples):
    n = len(samples)
    return 1.06 * np.std(samples) * n ** (-1/5)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, n_cur in zip(axes, sample_sizes):
    s = samples_by_n[n_cur]
    h_rot = silverman_bandwidth(s)
    f_hat = kde_estimate(x_grid, s, h_rot, gaussian_kernel)
    ax.plot(x_grid, f_true, label='true density', linewidth=2)
    ax.plot(x_grid, f_hat, label=f'KDE (h={h_rot:.3f})', linewidth=2)
    ax.hist(s, bins=30, density=True, alpha=0.25, label='histogram')
    ax.set_title(f'n = {n_cur}')
    ax.set_xlabel('x'); ax.set_ylabel('density')
    ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.show()

## d) Discussion — effect of sample size

- For $n = 100$ the estimate is noticeably noisy and a bit jagged. Both modes are detected, but their heights and the location of the minimum between them fluctuate.
- For $n = 500$ the estimate already approximates the true density quite well — the bimodal shape, the relative weights ($0.4 / 0.6$) and the asymmetric scale of the two components are all recovered.
- For $n = 2000$ the estimate is visually almost indistinguishable from the true density. The variance of the estimator drops, while the bias (controlled by the bandwidth) remains essentially the same.

This is exactly the **variance reduction** behaviour predicted by theory: $\mathrm{Var}\,\hat f(x) = O\!\big(\tfrac{1}{nh}\big)$.

# 2 / Influence of bandwidth

Using the sample with $n = 500$ and four bandwidths $h \in \{0.05, 0.2, 0.5, 1.0\}$.

In [ ]:
samples_500 = samples_by_n[500]
bandwidths = [0.05, 0.2, 0.5, 1.0]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, h_cur in zip(axes.ravel(), bandwidths):
    f_hat = kde_estimate(x_grid, samples_500, h_cur, gaussian_kernel)
    ax.plot(x_grid, f_true, label='true density', linewidth=2)
    ax.plot(x_grid, f_hat, label=f'KDE (h={h_cur})', linewidth=2)
    ax.set_title(f'h = {h_cur}')
    ax.grid(True, alpha=0.3); ax.legend()
for ax in axes[-1]:
    ax.set_xlabel('x')
for ax in axes[:, 0]:
    ax.set_ylabel('density')
plt.suptitle('Effect of bandwidth (n = 500, Gaussian kernel)', y=1.02)
plt.tight_layout()
plt.show()

### Discussion

- **$h = 0.05$ — strongly undersmoothed.** Each sample produces a narrow spike, so $\hat f$ is very rough and full of spurious local modes. Low bias, *very high variance*.
- **$h = 0.2$ — close to optimal.** Both modes are well separated, the asymmetric shape of the right component is preserved, and the estimate is smooth without losing structure.
- **$h = 0.5$ — slightly oversmoothed.** The two modes start to merge and their amplitudes are damped; the trough between them becomes shallow.
- **$h = 1.0$ — strongly oversmoothed.** The mixture collapses into an essentially unimodal bump; the bimodality is lost. *Very high bias*, low variance.

**Why does bandwidth matter more than the exact kernel?**
All sensible kernels (Gaussian, Epanechnikov, uniform, triangular) are symmetric, integrate to 1, and have similar overall shape. Differences between them mainly affect the local smoothness of $\hat f$. The *scale* over which neighbouring observations are averaged is set by $h$, and this scale directly controls how much fine structure is preserved or destroyed — so $h$ has a first-order effect on the estimate while the kernel only has a second-order one.

**Bias–variance tradeoff.**
Theory gives, locally,
$$\mathrm{Bias}\,\hat f(x) \propto h^2 f''(x),\qquad
\mathrm{Var}\,\hat f(x) \propto \frac{1}{nh}.$$
Small $h$ → small bias, large variance (the spiky picture). Large $h$ → small variance, large bias (the over-smoothed picture). The optimal $h$ minimises the sum (MISE) and shrinks at rate $h^\star \sim n^{-1/5}$.

# 3 / Compare kernels

Same $n = 500$ sample, a fixed bandwidth, three kernels.

In [ ]:
h_fixed = 0.3
kde_gauss = kde_estimate(x_grid, samples_500, h_fixed, gaussian_kernel)
kde_epan  = kde_estimate(x_grid, samples_500, h_fixed, epanechnikov_kernel)
kde_unif  = kde_estimate(x_grid, samples_500, h_fixed, uniform_kernel)
kde_tri   = kde_estimate(x_grid, samples_500, h_fixed, triangular_kernel)

plt.figure(figsize=(9, 5))
plt.plot(x_grid, f_true,   label='true density', linewidth=2, color='black')
plt.plot(x_grid, kde_gauss, label='Gaussian',     linewidth=1.6)
plt.plot(x_grid, kde_epan,  label='Epanechnikov', linewidth=1.6)
plt.plot(x_grid, kde_unif,  label='Uniform',      linewidth=1.6)
plt.plot(x_grid, kde_tri,   label='Triangular',   linewidth=1.6, linestyle='--')
plt.title(f'KDE — kernel comparison (n=500, h={h_fixed})')
plt.xlabel('x'); plt.ylabel('density')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

### Discussion

- **How different are the resulting estimates?**
  All four kernels track the true mixture density very closely. Differences are *small* and concentrated in the local roughness: the Gaussian estimate is smoothest, the Epanechnikov and triangular ones are slightly less smooth (their kernels have a kink at the boundary), and the uniform one is visibly the roughest because the rectangular kernel is discontinuous.
- **Which differences are caused by the kernel vs. the bandwidth?**
  The kernel mostly determines the *local smoothness* (continuity / differentiability) of $\hat f$, but does **not** change the overall scale or amount of averaging. The latter is set by $h$. Therefore the position of the modes, the location of the trough and the relative heights of the two bumps are essentially identical across kernels, while the *texture* of the curve differs.
- **Conclusion.**
  Kernel choice is a secondary design decision. Once you pick a reasonable kernel (e.g. Gaussian — smooth and convenient; or Epanechnikov — theoretically MISE-optimal for densities with bounded $f''$), the practical performance is governed by the bandwidth. In short: **spend your effort on $h$, not on $K$.**

# 4 / Approximate bandwidth selection

We numerically approximate
$$\mathrm{ISE}(h) \approx \int \big(\hat f_h(x) - f(x)\big)^2 dx$$
on a dense grid, sweep over $h$, and compare the minimiser with Silverman's rule of thumb.

In [ ]:
def approximate_ise(f_hat, f_true_vals, x_grid):
    return np.trapz((f_hat - f_true_vals)**2, x_grid)

x_dense = np.linspace(-5, 7, 1500)
f_true_dense = true_density(x_dense)

h_candidates = np.linspace(0.03, 1.5, 60)
ise_values = np.array([
    approximate_ise(kde_estimate(x_dense, samples_500, h, gaussian_kernel),
                    f_true_dense, x_dense)
    for h in h_candidates
])

h_star = h_candidates[np.argmin(ise_values)]
h_silv = silverman_bandwidth(samples_500)

print(f"ISE-optimal h*       = {h_star:.4f}")
print(f"Silverman rule h     = {h_silv:.4f}")
print(f"ISE at h*            = {ise_values.min():.5f}")
print(f"ISE at Silverman h   = {approximate_ise(kde_estimate(x_dense, samples_500, h_silv, gaussian_kernel), f_true_dense, x_dense):.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(h_candidates, ise_values, marker='o', markersize=3)
axes[0].axvline(h_star, color='red',  linestyle='--', label=f'h* = {h_star:.3f}')
axes[0].axvline(h_silv, color='green', linestyle='--', label=f'Silverman = {h_silv:.3f}')
axes[0].set_xlabel('bandwidth h'); axes[0].set_ylabel('approximate ISE')
axes[0].set_title('Approximate ISE as a function of h')
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(x_grid, f_true, label='true density', linewidth=2, color='black')
axes[1].plot(x_grid, kde_estimate(x_grid, samples_500, h_star, gaussian_kernel),
             label=f'KDE @ h* = {h_star:.3f}', linewidth=1.8)
axes[1].plot(x_grid, kde_estimate(x_grid, samples_500, h_silv, gaussian_kernel),
             label=f'KDE @ Silverman = {h_silv:.3f}', linewidth=1.8, linestyle='--')
axes[1].set_xlabel('x'); axes[1].set_ylabel('density')
axes[1].set_title('KDE at the two bandwidths')
axes[1].grid(True, alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.show()

### Discussion

- **Why is the optimal $h$ not the smallest possible one?**
  A very small $h$ gives almost zero bias but huge variance: each sample contributes a narrow spike, and the estimate fluctuates wildly between samples. The squared error is then dominated by variance, so ISE is large.
- **Why is it not the largest possible one?**
  A very large $h$ averages over so wide a window that real structure (here: the bimodality) is wiped out. Bias dominates and ISE blows up again.
- **Why is automatic bandwidth selection useful?**
  The ISE curve has a characteristic U-shape with a single, finite minimum that depends on $n$ and on the unknown $f$. Trying to find $h$ by eye is subjective and depends on the plot scale. Procedures like cross-validation, plug-in rules, or rules of thumb (Silverman) provide reproducible, data-driven choices.

Silverman's rule assumes $f$ is roughly Gaussian; here the truth is bimodal, so Silverman tends to *over-smooth* slightly compared to the true ISE-optimal $h^\star$. Still, it lands in the right ballpark and is a very useful default.

# 5 / Density-based classification using KDE

Two classes in $\mathbb{R}$:

$$\omega_1 \sim 0.5\,\mathcal{N}(-2, 0.6^2) + 0.5\,\mathcal{N}(0, 0.7^2),$$
$$\omega_2 \sim 0.4\,\mathcal{N}(1.5, 0.5^2) + 0.6\,\mathcal{N}(3, 0.8^2),$$

with equal priors $P(\omega_1) = P(\omega_2) = 0.5$.

Classifier: $\hat y(x) = \arg\max_i\ \hat p(x\mid \omega_i)\,P(\omega_i)$.

## a) Generate 400 samples from each class

In [ ]:
rng = np.random.default_rng(7)

def sample_class1(n, rng):
    z = rng.choice([0, 1], size=n, p=[0.5, 0.5])
    x = np.empty(n)
    x[z == 0] = rng.normal(-2.0, 0.6, size=np.sum(z == 0))
    x[z == 1] = rng.normal( 0.0, 0.7, size=np.sum(z == 1))
    return x

def sample_class2(n, rng):
    z = rng.choice([0, 1], size=n, p=[0.4, 0.6])
    x = np.empty(n)
    x[z == 0] = rng.normal(1.5, 0.5, size=np.sum(z == 0))
    x[z == 1] = rng.normal(3.0, 0.8, size=np.sum(z == 1))
    return x

def true_density_class1(x):
    return 0.5 * gaussian_pdf_1d(x, -2.0, 0.6) + 0.5 * gaussian_pdf_1d(x, 0.0, 0.7)

def true_density_class2(x):
    return 0.4 * gaussian_pdf_1d(x, 1.5, 0.5) + 0.6 * gaussian_pdf_1d(x, 3.0, 0.8)

n_per_class = 400
X1_train = sample_class1(n_per_class, rng)
X2_train = sample_class2(n_per_class, rng)

X1_test  = sample_class1(n_per_class, rng)
X2_test  = sample_class2(n_per_class, rng)

X_test = np.concatenate([X1_test, X2_test])
y_test = np.array([1] * n_per_class + [2] * n_per_class)

print("Class 1 train mean / std:", X1_train.mean(), X1_train.std())
print("Class 2 train mean / std:", X2_train.mean(), X2_train.std())

## b) Estimate class-conditional densities with KDE

We use a Gaussian kernel and Silverman's rule for each class independently.

In [ ]:
h1 = silverman_bandwidth(X1_train)
h2 = silverman_bandwidth(X2_train)
print(f"h1 = {h1:.3f}    h2 = {h2:.3f}")

x_grid_cls = np.linspace(-5, 6, 600)
p_hat_1 = kde_estimate(x_grid_cls, X1_train, h1, gaussian_kernel)
p_hat_2 = kde_estimate(x_grid_cls, X2_train, h2, gaussian_kernel)

## c) Construct the classifier

$$\hat y(x) = \arg\max_i\ \hat p(x\mid \omega_i)\,P(\omega_i).$$

In [ ]:
P1 = 0.5
P2 = 0.5

def kde_classifier(x_eval, X1, X2, h1, h2, P1, P2, kernel_fn=gaussian_kernel):
    x_eval = np.atleast_1d(x_eval)
    p1 = kde_estimate(x_eval, X1, h1, kernel_fn) * P1
    p2 = kde_estimate(x_eval, X2, h2, kernel_fn) * P2
    return np.where(p1 >= p2, 1, 2)

## d) Evaluate on a separate test set

In [ ]:
y_pred = kde_classifier(X_test, X1_train, X2_train, h1, h2, P1, P2)

def confusion_matrix_2x2(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=int)
    for t in (1, 2):
        for p in (1, 2):
            cm[t-1, p-1] = np.sum((y_true == t) & (y_pred == p))
    return cm

cm = confusion_matrix_2x2(y_test, y_pred)
acc = (y_test == y_pred).mean()
err = 1 - acc

print("Confusion matrix (rows = T, cols = P):")
print("          pred=1   pred=2")
print(f"true=1   {cm[0,0]:>6}   {cm[0,1]:>6}")
print(f"true=2   {cm[1,0]:>6}   {cm[1,1]:>6}")
print(f"Accuracy:              {acc:.4f}")
print(f"Empirical error rate:  {err:.4f}")

## e) Plot estimated densities and classification regions

In [ ]:
preds_grid = kde_classifier(x_grid_cls, X1_train, X2_train, h1, h2, P1, P2)
f1_true = true_density_class1(x_grid_cls)
f2_true = true_density_class2(x_grid_cls)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (left) estimated class densities
ax = axes[0]
ax.plot(x_grid_cls, p_hat_1, label=r'$\hat p(x\mid \omega_1)$', linewidth=2)
ax.plot(x_grid_cls, p_hat_2, label=r'$\hat p(x\mid \omega_2)$', linewidth=2)
ax.plot(x_grid_cls, f1_true, label=r'true $p(x\mid \omega_1)$', linewidth=1, linestyle='--', color='C0')
ax.plot(x_grid_cls, f2_true, label=r'true $p(x\mid \omega_2)$', linewidth=1, linestyle='--', color='C1')
ax.set_xlabel('x'); ax.set_ylabel('density')
ax.set_title('Estimated vs true class-conditional densities')
ax.grid(True, alpha=0.3); ax.legend()

# (right) classification regions
ax = axes[1]
ax.fill_between(x_grid_cls, 0, p_hat_1.max() * 1.1,
                where=(preds_grid == 1), alpha=0.15, color='C0', label=r'region $\omega_1$')
ax.fill_between(x_grid_cls, 0, p_hat_2.max() * 1.1,
                where=(preds_grid == 2), alpha=0.15, color='C1', label=r'region $\omega_2$')
ax.plot(x_grid_cls, p_hat_1 * P1, label=r'$\hat p(x\mid \omega_1)P(\omega_1)$', linewidth=2)
ax.plot(x_grid_cls, p_hat_2 * P2, label=r'$\hat p(x\mid \omega_2)P(\omega_2)$', linewidth=2)
ax.scatter(X1_train, np.full_like(X1_train, -0.01), s=10, marker='|', color='C0', alpha=0.5)
ax.scatter(X2_train, np.full_like(X2_train, -0.02), s=10, marker='|', color='C1', alpha=0.5)
ax.set_xlabel('x'); ax.set_ylabel('weighted density')
ax.set_title('Classification regions (KDE-based Bayes rule)')
ax.grid(True, alpha=0.3); ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

### Discussion

- **Why can KDE be used for classification?**
  The Bayes-optimal decision rule depends on the class-conditional densities $p(x\mid\omega_i)$. KDE provides a nonparametric estimate of each, so plugging $\hat p(x\mid\omega_i)$ into the Bayes rule gives a perfectly valid (plug-in) classifier — as $n\to\infty$ and $h\to 0$ at the right rate, $\hat p \to p$ and the classifier approaches the Bayes-optimal one.
- **Strengths.**
  No parametric assumption on the class densities — works for skewed, multimodal, or otherwise non-Gaussian distributions (exactly the case here, where each class is itself a mixture). Simple to implement, easy to inspect (we *see* the estimated densities), and naturally handles complex decision boundaries.
- **Weaknesses compared with a parametric classifier (LDA / QDA / Gaussian mixture with known structure):**
  (i) higher variance for the same $n$ — KDE pays for its flexibility;
  (ii) sensitive to the bandwidth, which must be selected carefully and may differ per class;
  (iii) suffers strongly from the curse of dimensionality — required sample size grows roughly as $n \sim h^{-d}$;
  (iv) prediction is $O(n)$ per query point because the training set must be kept around (a lazy method).
  When the parametric model is correct (or close to it), a parametric classifier is more sample-efficient. KDE is the safer choice when you do *not* trust the parametric form.

-----
## How do they do it in practice — scikit-learn / scipy

Quick cross-check: same bandwidth, same data, library implementation.

In [ ]:
from sklearn.neighbors import KernelDensity

# Task 1 — KDE on the n=500 sample
kde_sk = KernelDensity(kernel='gaussian',
                       bandwidth=silverman_bandwidth(samples_500))
kde_sk.fit(samples_500[:, None])
f_sk = np.exp(kde_sk.score_samples(x_grid[:, None]))

plt.figure(figsize=(8, 4))
plt.plot(x_grid, f_true, label='true density', linewidth=2)
plt.plot(x_grid, f_sk,    label='sklearn KernelDensity', linewidth=2, linestyle='--')
plt.title('sklearn KDE vs from-scratch implementation')
plt.xlabel('x'); plt.ylabel('density')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

# Task 5 — classifier accuracy with sklearn KDEs
kde1_sk = KernelDensity(kernel='gaussian', bandwidth=h1).fit(X1_train[:, None])
kde2_sk = KernelDensity(kernel='gaussian', bandwidth=h2).fit(X2_train[:, None])
log_p1 = kde1_sk.score_samples(X_test[:, None]) + np.log(P1)
log_p2 = kde2_sk.score_samples(X_test[:, None]) + np.log(P2)
y_pred_sk = np.where(log_p1 >= log_p2, 1, 2)
print("Test accuracy (sklearn KDE classifier):", (y_test == y_pred_sk).mean())